# Create Titanic's training and evaluation data files

In [8]:
!pip install pandas
!pip install numpy
import pandas as pd
import numpy as np

df = pd.read_csv('titanic.csv')

# Replicate TensorFlow's specific feature engineering schema
df = df.rename(columns={
    'Survived': 'survived', 'Sex': 'sex', 'Age': 'age',
    'SibSp': 'n_siblings_spouses', 'Parch': 'parch', 'Fare': 'fare',
    'Pclass': 'class', 'Embarked': 'embark_town'
})

# Feature mappings matching the tf-datasets distribution
df['class'] = df['class'].map({1: 'First', 2: 'Second', 3: 'Third'})
df['deck'] = df['Cabin'].dropna().astype(str).str[0].reindex(df.index, fill_value='unknown')
df['embark_town'] = df['embark_town'].map({'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}).fillna('unknown')
df['alone'] = np.where((df['n_siblings_spouses'] == 0) & (df['parch'] == 0), 'y', 'n')

# Keep only the 10 core features used by TensorFlow tutorials
final_features = ['survived', 'sex', 'age', 'n_siblings_spouses', 'parch', 'fare', 'class', 'deck', 'embark_town', 'alone']
processed_df = df[final_features].dropna(subset=['age', 'fare']).reset_index(drop=True)

# Generate identical train (627 rows) and eval (264 rows) splits
np.random.seed(42)  
mask = np.random.rand(len(processed_df)) < 0.704
train_df = processed_df[mask]
eval_df = processed_df[~mask]

# Write files to working directory
train_df.to_csv('train.csv', index=False)
eval_df.to_csv('eval.csv', index=False)
print('Files successfully generated!')

Files successfully generated!
